In [ ]:
# Papermill parameters — injected by Airflow at runtime
import json

SQL_CONFIG = {}   # injected: {"USER":..., "PASSWORD":..., "ACCOUNT":..., "WAREHOUSE":..., "ROLE":...}
START_DATE = "rolling_35_day"  # overridden by Airflow; set manually for backfills e.g. "2025-03-01"
RUN_DATE = "2025-01-01"        # execution date, used for output filename
ENV = "dev"                    # "prod" or "dev" — controls output table
dag_run_id = "manual"
execution_date = "2025-01-01"


In [ ]:
import pandas as pd
import numpy as np
import time as time_lib
import os
import warnings
from datetime import datetime
from sqlalchemy import create_engine
from ChannelAttribution import markov_model, heuristic_models

warnings.filterwarnings("ignore")
print(f"Run date: {RUN_DATE} | Start date: {START_DATE} | ENV: {ENV}")


In [ ]:
# Build Snowflake engine from injected SQL_CONFIG
if isinstance(SQL_CONFIG, str):
    SQL_CONFIG = json.loads(SQL_CONFIG)

engine = create_engine(
    "snowflake://{USER}:{PASSWORD}@{ACCOUNT}/?warehouse={WAREHOUSE}&role={ROLE}".format(**SQL_CONFIG)
)
print("Snowflake connection established")


In [ ]:
q = f"""
with conv_df as (
    select
        mta_order_type
        , case
            when i.unified_attribution_key_touchpoints = '' then 'Missing from Rudderstack'
            when i.unified_attribution_key_touchpoints is null then 'Missing from Rudderstack'
            else i.unified_attribution_key_touchpoints end as unified_attribution_key_touchpoints
        , i.marketing_channel_touchpoints
        , date_trunc('day', a.order_created_date) as order_date
        , a.unified_region as region
        , count(distinct a.order_id) as converted
    from prod_edg.mart_analytics.analytics_order a
    left join prod_edg.mart_analytics.mta_all_models_collected_user_touchpoints i
        on a.order_id = i.order_id
    where a.order_created_date >= '{START_DATE}'
      and a.sales_channel = 'DTC'
      and i.mta_order_type in ('agz', 'winback')
    group by all
),
agp_person_ids as (
    select distinct
        analytics_web_user_journey.ag_person_id,
        analytics_order.customer_id,
        analytics_order.unified_region as region
    from prod_edg.mart_analytics.analytics_web_user_journey analytics_web_user_journey
    left join prod_edg.mart_analytics.analytics_order
        on analytics_web_user_journey.order_id = analytics_order.order_id
    where analytics_web_user_journey.order_id is not null
),
non_conv_df as (
    select
        'winback' as mta_order_type,
        case
            when i.unified_attribution_key_touchpoints = '' then 'Missing from Rudderstack'
            when i.unified_attribution_key_touchpoints is null then 'Missing from Rudderstack'
            else i.unified_attribution_key_touchpoints end as unified_attribution_key_touchpoints
        , i.marketing_channel_touchpoints
        , date_trunc('day', i.last_touch_date) as order_date
        , a.region
        , count(distinct i.ag_person_id) as non_converted
    from prod_edg.mart_analytics.mta_winback_non_conv_with_collected_touchpoints i
    left join agp_person_ids a on i.ag_person_id = a.ag_person_id
    where i.last_touch_date >= '{START_DATE}'
      and a.region is not null
      and i.last_touch_date is not null
    group by all
),
joined as (
    select mta_order_type, unified_attribution_key_touchpoints, marketing_channel_touchpoints,
           order_date, region, converted, 0 as non_converted from conv_df
    union
    select mta_order_type, unified_attribution_key_touchpoints, marketing_channel_touchpoints,
           order_date, region, 0 as converted, non_converted from non_conv_df
),
agg as (
    select mta_order_type, unified_attribution_key_touchpoints, marketing_channel_touchpoints,
           order_date, region, sum(converted) as converted, sum(non_converted) as non_converted
    from joined group by 1,2,3,4,5
)
select *,
    case when unified_attribution_key_touchpoints ilike '%missing from%' then 1 else 0 end as mfr
from agg
"""

df = pd.read_sql(q, engine)
print(f"Extracted {len(df):,} rows")
assert not df.empty, "No data returned — check START_DATE and source tables"


In [ ]:
# Clean MFR strings
df["unified_attribution_key_touchpoints"] = (
    df["unified_attribution_key_touchpoints"]
    .astype(str)
    .replace({"": "Missing from Rudderstack", "None": "Missing from Rudderstack"})
)
df["mfr"] = df["unified_attribution_key_touchpoints"].str.contains("Missing from", na=False).astype(int)

# Build direct-only flag and trim trailing Direct from paths
uniqs = df[["unified_attribution_key_touchpoints"]].drop_duplicates().reset_index(drop=True)
uniqs["split_tps"] = uniqs["unified_attribution_key_touchpoints"].str.split(" > ")
uniqs["uniq"] = uniqs["split_tps"].apply(lambda x: list(set(x)))
uniqs["lenz"] = uniqs["uniq"].apply(len)
uniqs["direct_only"] = uniqs.apply(lambda r: int(r["lenz"] == 1 and r["uniq"][0] == "Direct"), axis=1)

def trim_trailing_direct(row):
    if row["direct_only"] == 1:
        return row["unified_attribution_key_touchpoints"]
    tps = row["split_tps"]
    non_direct = [x for x in tps if x != "Direct"]
    if not non_direct:
        return row["unified_attribution_key_touchpoints"]
    max_idx = max(i for i, v in enumerate(tps) if v == non_direct[-1])
    return " > ".join(tps[:max_idx + 1])

uniqs["unified_attribution_key_touchpoints2"] = uniqs.apply(trim_trailing_direct, axis=1)
df = df.merge(uniqs[["unified_attribution_key_touchpoints", "unified_attribution_key_touchpoints2", "direct_only"]])
df = df.drop(columns=["unified_attribution_key_touchpoints"]).rename(
    columns={"unified_attribution_key_touchpoints2": "unified_attribution_key_touchpoints"}
)
print(f"Clean complete: {len(df):,} rows")


In [ ]:
def run_markov_granularity(df, grouped_touchpoints, grouping, time_col, mfr_col):
    markov_dfs = []
    for date in df.sort_values(by="order_date")[time_col].unique():
        for region in df["region"].unique():
            for mta_order_type in df["mta_order_type"].unique():
                try:
                    month_data = (
                        df[df[time_col] == date]
                        .query(f'region=="{region}"')
                        .query(f'mta_order_type=="{mta_order_type}"')
                    )
                    mfr_data    = month_data[month_data[mfr_col] == 1]
                    direct_data = month_data[month_data["direct_only"] == 1]
                    month_data  = month_data[(month_data[mfr_col] == 0) & (month_data["direct_only"] == 0)]

                    path_data = (
                        month_data.groupby(grouped_touchpoints)
                        .agg({"converted": "sum", "non_converted": "sum"})
                        .reset_index()
                    )
                    path_data[grouped_touchpoints] = path_data[grouped_touchpoints].str.replace("Direct > ", "", regex=False)
                    paths = path_data.sort_values(by="converted", ascending=False).reset_index(drop=True)
                    paths.columns = [grouping, "total_conversions", "total_null"]
                    paths["total_conversion_value"] = 1

                    M = markov_model(paths, grouping, "total_conversions",
                                    var_value="total_conversion_value", var_null="total_null", verbose=False)
                    H = heuristic_models(paths, grouping, "total_conversions", var_value="total_conversion_value")

                    mh = M.merge(H)[
                        ["channel_name", "total_conversions", "first_touch_conversions",
                         "last_touch_conversions", "linear_touch_conversions"]
                    ].rename(columns={"total_conversions": "markov_conversions", "channel_name": grouping})

                    for label, src in [("Missing from Rudderstack", mfr_data), ("Direct", direct_data)]:
                        extra = pd.DataFrame(
                            [[label, src["converted"].sum()] + [src["converted"].sum()] * 3],
                            columns=mh.columns
                        )
                        mh = pd.concat([mh, extra], ignore_index=True)

                    R = markov_model(paths, grouping, "total_conversions",
                                    var_value="total_conversion_value", var_null="total_null",
                                    out_more=True, verbose=False)
                    R = (R["removal_effects"][["channel_name", "removal_effects_conversion"]]
                         .rename(columns={"removal_effects_conversion": "removal_effects", "channel_name": grouping}))
                    mh = mh.merge(R, how="left")
                    mh[time_col] = date
                    mh["region"] = region
                    mh["mta_order_type"] = mta_order_type
                    markov_dfs.append(mh)
                except Exception as e:
                    print(f"  skipped {date}/{region}/{mta_order_type}: {e}")
        print(f"{date} done")
    return pd.concat(markov_dfs).reset_index(drop=True)

start_time = time_lib.perf_counter()
markov_cn = run_markov_granularity(
    df,
    grouped_touchpoints="unified_attribution_key_touchpoints",
    grouping="unified_attribution_key",
    time_col="order_date",
    mfr_col="mfr"
)
elapsed_hours = (time_lib.perf_counter() - start_time) / 3600
print(f"{elapsed_hours:.4f} hours | {len(markov_cn):,} output rows")


In [ ]:
# ── Review outcomes: compare markov_conversions vs raw orders ──
check_q = f"""
(
    select 'winback' as mta_order_type,
           date(date_trunc('month', a.order_created_ts)) as month,
           count(distinct order_id) as raw_orders
    from prod_edg.mart_analytics.analytics_order a
    where date_trunc('day', a.order_created_ts) >= '{START_DATE}'
      and a.unified_region = 'NA' and a.sales_channel = 'DTC'
      and is_winback_order = true
    group by all
)
union
(
    select 'agz' as mta_order_type,
           date(date_trunc('month', a.order_created_ts)) as month,
           count(distinct order_id) as raw_orders
    from prod_edg.mart_analytics.analytics_order a
    where date_trunc('day', a.order_created_ts) >= '{START_DATE}'
      and a.unified_region = 'NA' and a.sales_channel = 'DTC'
      and is_first_agz_order = true
    group by all
)
"""

raw_check = pd.read_sql(check_q, engine)
raw_check["year"]  = pd.to_datetime(raw_check["month"]).dt.year
raw_check["month"] = pd.to_datetime(raw_check["month"]).dt.month

m2 = markov_cn.copy()
m2["month"] = pd.to_datetime(m2["order_date"]).dt.month
m2["year"]  = pd.to_datetime(m2["order_date"]).dt.year

val_check = (
    m2.query('region=="NA"').groupby(["mta_order_type", "month", "year"])["markov_conversions"]
    .sum().reset_index().sort_values("month")
)
vcm = val_check.merge(raw_check)
vcm["diff"] = vcm["markov_conversions"] - vcm["raw_orders"]
vcm["pct_diff"] = (vcm["diff"] / vcm["raw_orders"] * 100).round(2)

print(vcm.to_string(index=False))

# Post to Slack
import urllib.request

slack_webhook = os.environ.get("SLACK_WEBHOOK_URL", "")
if slack_webhook:
    has_discrepancy = (vcm["diff"] != 0).any()
    status = ":warning:" if has_discrepancy else ":white_check_mark:"
    rows = []
    for _, r in vcm.sort_values(["year", "month"]).iterrows():
        flag = " :warning:" if r["diff"] != 0 else ""
        rows.append(f"  {int(r['year'])}-{int(r['month']):02d}  {r['mta_order_type']:<8}  "
                    f"markov: {r['markov_conversions']:>8,.0f}  raw: {r['raw_orders']:>8,}  diff: {r['diff']:>+.0f}{flag}")
    table_str = "\n".join(rows)
    msg = (
        f"{status} *MTA AGZ/Winback — Review Outcomes* ({RUN_DATE})\n"
        f"\n"
        + (":warning: *Discrepancies found — review before relying on output.*"
           if has_discrepancy else ":white_check_mark: All diffs are 0 — looks good.")
    )
    payload = json.dumps({"text": msg}).encode()
    req = urllib.request.Request(slack_webhook, data=payload, headers={"Content-Type": "application/json"})
    urllib.request.urlopen(req)
    print(f"Slack message sent ({status})")
else:
    print("No SLACK_WEBHOOK_URL set — skipping Slack notification")


In [ ]:
# Drop removal_effects before writing (not needed downstream)
to_upload = markov_cn.drop(columns=["removal_effects"], errors="ignore").copy()
to_upload["_last_updated"] = RUN_DATE

# SageMaker mounts the output dir at /opt/ml/processing/output
# dag_factory_blocks.prepare_merge_metadata looks for: markov_attrkeydaily_{RUN_DATE}.csv
output_dir = "/opt/ml/processing/output"
os.makedirs(output_dir, exist_ok=True)
output_filename = f"markov_attrkeydaily_{RUN_DATE}.csv"
output_path = os.path.join(output_dir, output_filename)

to_upload.to_csv(output_path, index=False)
print(f"Written {len(to_upload):,} rows to {output_path}")
